# Tutorial: Deploying a PySpark Workload on Google Cloud with Terraform and Docker

This tutorial walks you through provisioning a complete cloud environment that can build, deploy, and run a containerized data-processing job from scratch. The job itself — credit-card fraud detection with PySpark — is small on purpose. The interesting part is everything *around* it: the network, the storage, the identity, the bootstrap, and how Terraform glues it all together.

By the end you will have a single command (`terraform apply`) that creates a private network, uploads your dataset, builds and runs a Docker container, and writes results back to disk on a virtual machine — and a single command (`terraform destroy`) that cleans it all up.


## Learning objectives

After completing this tutorial you should be able to:

- **C1** — Explain why Infrastructure as Code (IaC) makes cloud deployments reproducible.
- **C2** — Describe how Terraform, Docker, and a Compute Engine VM compose into a single deployable unit.
- **C3** — Identify the trust boundary between a VM's identity (service account) and the resources it consumes (GCS bucket).
- **P1** — Write a multi-file Terraform configuration that provisions a custom VPC, subnet, firewall rule, GCS bucket, IAM binding, and Compute Engine instance.
- **P2** — Use the GCE metadata server to obtain an OAuth token and download a Cloud Storage object from inside a VM.
- **P3** — Bootstrap Docker on a fresh Ubuntu VM and run a containerized PySpark job non-interactively.


## Why this tutorial exists

Running a one-off PySpark script on your laptop is easy. Running the *same* script in a way that a colleague — or your future self six months from now — can reproduce is much harder. You need the same Python version, the same Java runtime (PySpark 4 requires JDK 17), the same dataset in the same place, and the same network conditions. Cloud computing solves this by treating the entire environment as code.

This tutorial maps to several lectures in the module:

- **Infrastructure as Code Fundamentals (Session 01_02)** — the `init / plan / apply / destroy` lifecycle.
- **Managing GCP Resources (Session 02_02)** — custom VPCs, firewall rules, data sources, implicit dependencies.
- **Variables & Configuration (Session 03_02)** — `variables.tf`, `outputs.tf`, `terraform.tfvars`.
- **Docker Fundamentals & Custom Images (Sessions 01_01 / 02_01)** — Dockerfiles, layer caching, `.dockerignore`.
- **PySpark Parts 1, 2, 4 (Sessions 01_03 / 02_03 / 04_03)** — `SparkSession`, DataFrames, cleaning, `spark.ml` pipelines.


## What we are building

```
                                                              .─────────────.
   developer's laptop                                        ( your GCP     )
   ┌───────────────────────┐    terraform apply               (  project    )
   │  Tutorial1/           │ ─────────────────────────►       '─────────────'
   │   data/creditcard.csv │                                         │
   │   docker/Dockerfile   │                                         │
   │   scripts/*.py        │           ┌─────────────────────────────┴──────────┐
   │   infra/terraform/    │           │  custom VPC: fraud-detection-vpc       │
   └───────────────────────┘           │   └─ subnet: fraud-public-subnet       │
                                       │       └─ firewall: allow_ssh           │
                                       │                                        │
                                       │  GCS bucket: <project>-fraud-dataset   │
                                       │   └─ object: creditcard.csv            │
                                       │                                        │
                                       │  IAM:                                  │
                                       │   service account: fraud-vm-sa         │
                                       │   role: storage.objectViewer (bucket)  │
                                       │                                        │
                                       │  Compute Engine VM: fraud-vm-1         │
                                       │   ├─ uses fraud-vm-sa                  │
                                       │   ├─ runs startup.sh on first boot     │
                                       │   │   ├─ apt install docker.io         │
                                       │   │   ├─ writes Dockerfile, script     │
                                       │   │   ├─ curl dataset from GCS         │
                                       │   │   └─ build + run container         │
                                       │   └─ writes /opt/vm-ml-tutorial/output │
                                       └────────────────────────────────────────┘
```


## Prerequisites

- A Google Cloud account with billing enabled, and a project ID you can deploy into.
- The `gcloud` CLI installed and authenticated: `gcloud auth application-default login`.
- Terraform `>= 1.5.0`.
- The Kaggle [creditcard.csv](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) dataset, placed at `data/creditcard.csv` in this repo. It is gitignored and will be uploaded to GCS by Terraform.
- Optional, for local testing only: Docker Desktop and JDK 17.

> **Cost note.** This tutorial provisions one `e2-medium` VM, one regional GCS bucket, and small networking resources. At time of writing this is well within the GCP free tier for short experiments. Always run `terraform destroy` when you are finished.


## Step 0 — The ML payload in 60 seconds

We are packaging a binary classifier that flags fraudulent credit-card transactions. The dataset (Kaggle ULB) has 284,807 transactions, 30 numeric predictors, and an extremely imbalanced label (`Class = 1` for fraud, occurring in roughly 0.17% of rows). The model is `pyspark.ml.classification.LogisticRegression` wrapped in a `Pipeline` with a `VectorAssembler`, evaluated by `BinaryClassificationEvaluator(metricName="areaUnderROC")`.

The full pipeline lives in [scripts/fraud_detection_pyspark.py](scripts/fraud_detection_pyspark.py). The exact same code is the basis for the (forthcoming) `tutorial.ipynb` walkthrough that explains the data-science choices in detail.

**For the purposes of this tutorial, treat the script as a black box** that:

- Reads `data/creditcard.csv`.
- Writes a JSON summary to `output/summary.json` with row counts, AUC, and a confusion matrix.
- Writes 50 sample predictions to `output/sample_predictions.csv`.

Our job for the rest of the tutorial is to make sure that script runs reliably in a clean cloud environment.


## Step 1 — Containerize the workload

The first reproducibility problem is the runtime. PySpark 4 needs JDK 17, but Ubuntu 22.04 ships with a different Java version, and your laptop probably has yet another. The fix is a Docker image that pins everything.

Open [docker/Dockerfile](docker/Dockerfile):

```dockerfile
# Start small python image
FROM python:3.12-slim

# Set the working directory in the container (create app directory)
WORKDIR /app

# Install Java (required for PySpark)
RUN apt-get update && \
    apt-get install -y --no-install-recommends openjdk-17-jre-headless && \
    rm -rf /var/lib/apt/lists/*

# Assign environment variables for Java and Spark
ENV JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
ENV SPARK_LOCAL_HOSTNAME=localhost

# Install python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt

# Copy the application code to the container and create necessary directories
COPY scripts ./scripts
RUN mkdir -p data output

CMD ["python", "scripts/fraud_detection_pyspark.py", "--data-path", "data/creditcard.csv", "--output-dir", "output", "--max-iter", "10"]
```

Three things here are worth noting because they map directly onto Session 02_01 (Custom Images):

1. **Layer ordering for cache efficiency.** `requirements.txt` is copied and installed *before* application code. If you only edit your Python script, Docker reuses the cached `pip install` layer and rebuilds in seconds instead of minutes.
2. **`--no-install-recommends` and `rm -rf /var/lib/apt/lists/*`.** Standard tricks to keep the image small. The lecture's "Multi-Stage Builds" pattern (slide 02_01 p25) would shrink it further; that is a natural follow-up.
3. **Environment variables.** `JAVA_HOME` is what PySpark consults at runtime. `SPARK_LOCAL_HOSTNAME=localhost` avoids hostname-resolution warnings when Spark runs in container-only mode.

The companion [docker/.dockerignore](docker/.dockerignore) keeps the dataset, notebook, git history, and Terraform state out of the build context — without it, every `docker build` would ship 150 MB of CSV to the daemon.

> **Test it locally (optional).** If you have JDK 17 installed, you can run the script directly: `python scripts/fraud_detection_pyspark.py --data-path output/smoke_creditcard.csv --output-dir output/smoke_run --max-iter 2`. The `output/smoke_creditcard.csv` fixture is a 200-row sample for fast iteration.


## Step 2 — Define the private network

Real workloads should not live on the default network. Session 02_02 explicitly recommends creating a custom-mode VPC. Open [infra/terraform/main.tf](infra/terraform/main.tf):

```hcl
# Create a custom VPC network
resource "google_compute_network" "custom_vpc" {
  name                    = var.vpc_name
  auto_create_subnetworks = false
}

# Creates a public subnet in the custom VPC
resource "google_compute_subnetwork" "public_subnet" {
  name          = var.subnet_name
  ip_cidr_range = var.subnet_cidr_range
  region        = var.region
  network       = google_compute_network.custom_vpc.id
}

# Create a firewall rule to allow SSH access to the VM
resource "google_compute_firewall" "allow_ssh" {
  name        = "${var.instance_name}-allow-ssh"
  network     = google_compute_network.custom_vpc.name

  allow {
    protocol = "tcp"
    ports    = ["22"]
  }

  source_ranges = var.ssh_source_ranges
  target_tags   = ["ssh-enabled"]
}
```

Three concepts:

- **`auto_create_subnetworks = false`** is the GCP best practice. The default network gives you 35 pre-made subnets you do not control. Custom mode means *you* decide the IP ranges.
- **Implicit dependencies.** `network = google_compute_network.custom_vpc.id` is a cross-resource reference. Terraform's graph-builder reads this and automatically creates the VPC before the subnet, the subnet before anything that uses it. You never write "create A, then B, then C" — you describe the desired state and Terraform infers the order. This is the *declarative* model from Session 01_02 slide 6.
- **Firewall rules with `target_tags`.** Instead of attaching the rule to specific VMs, we tag the rule (`target_tags = ["ssh-enabled"]`) and tag the VMs we want it to apply to. This scales: add another VM with the `ssh-enabled` tag and it inherits the rule for free.

> **Security callout.** The default `ssh_source_ranges = ["0.0.0.0/0"]` opens SSH to the world. Override it in `terraform.tfvars` with your own public IP in CIDR form (`["X.X.X.X/32"]`) before applying.


## Step 3 — Stage the dataset in Cloud Storage

The dataset is too big to embed in the startup script as a heredoc, and `scp`-ing it manually defeats the "one command to deploy" goal. We let Terraform upload it during `apply`. Open [infra/terraform/storage.tf](infra/terraform/storage.tf):

```hcl
# Derive a bucket name for the automated fraud dataset upload
locals {
  dataset_bucket_name = var.dataset_bucket_name != null ? var.dataset_bucket_name : "${var.project_id}-fraud-dataset"
}

# Create a GCS bucket to hold the fraud dataset for automated VM downloads
resource "google_storage_bucket" "dataset" {
  name                        = local.dataset_bucket_name
  location                    = var.region
  force_destroy               = true
  uniform_bucket_level_access = true

  labels = {
    tutorial = "fraud-dataset"
    course   = "cloud-hpc"
  }
}

# Upload the local credit card fraud dataset into the GCS bucket
resource "google_storage_bucket_object" "creditcard_dataset" {
  name   = var.dataset_object_name
  bucket = google_storage_bucket.dataset.name
  source = "${path.module}/../../data/creditcard.csv"
}
```

What this is doing:

- **`locals { dataset_bucket_name = ... }`** lets the user override the bucket name with a variable, but defaults to `<project>-fraud-dataset`. GCS bucket names are globally unique across all of Google Cloud, so prefixing with the project ID is a safe default.
- **`uniform_bucket_level_access = true`** disables per-object ACLs. Modern GCP best practice — it forces all access to go through IAM, which is what we control in the next step.
- **`force_destroy = true`** lets `terraform destroy` delete the bucket even if it contains objects. Convenient for tutorials, *dangerous in production*.
- **`google_storage_bucket_object`** with `source = "../../data/creditcard.csv"` does the actual upload during `terraform apply`. Terraform reads the file from the Terraform module's filesystem and streams it to GCS.


## Step 4 — Give the VM an identity

The VM needs to download the dataset from the bucket. The lazy approach is to make the bucket public. The right approach is to give the VM its own service account and grant *that account* read-only access to *that bucket*. Open [infra/terraform/iam.tf](infra/terraform/iam.tf):

```hcl
# Create a VM service account that can read the dataset bucket
resource "google_service_account" "vm_dataset_reader" {
  account_id   = var.vm_service_account_id
  display_name = "Fraud Detection VM Dataset Reader"
}

# Grant the VM service account read access to the dataset bucket
resource "google_storage_bucket_iam_member" "dataset_reader" {
  bucket = google_storage_bucket.dataset.name
  role   = "roles/storage.objectViewer"
  member = "serviceAccount:${google_service_account.vm_dataset_reader.email}"
}
```

Two concepts:

- **Principle of least privilege.** `roles/storage.objectViewer` lets the VM read objects from this one bucket and nothing else. It cannot list other buckets, write objects, or change permissions. If the VM is ever compromised, the blast radius is "an attacker can read public-grade fraud data" — not "an attacker has cloud-admin powers".
- **Scoped IAM binding.** `google_storage_bucket_iam_member` attaches the role *to the bucket*, not project-wide. The same service account on a different bucket would have no rights. This is much safer than `google_project_iam_member`, which would grant the role on every bucket in the project.


## Step 5 — Provision the VM and its startup script

This is where everything comes together. Back in [infra/terraform/main.tf](infra/terraform/main.tf):

```hcl
# Fetch the latest Ubuntu 22.04 LTS image from the public image family
data "google_compute_image" "ubuntu" {
  family  = "ubuntu-2204-lts"
  project = "ubuntu-os-cloud"
}

resource "google_compute_instance" "ml_vm" {
  name         = var.instance_name
  machine_type = var.machine_type
  zone         = var.zone
  tags         = ["ssh-enabled", "terraform-tutorial"]

  boot_disk {
    initialize_params {
      image = data.google_compute_image.ubuntu.self_link
      size  = 20
      type  = "pd-balanced"
    }
  }

  network_interface {
    subnetwork = google_compute_subnetwork.public_subnet.id
    access_config {}
  }

  service_account {
    email  = google_service_account.vm_dataset_reader.email
    scopes = ["https://www.googleapis.com/auth/devstorage.read_only"]
  }

  metadata_startup_script = templatefile("${path.module}/startup.sh", {
    dockerfile_txt      = file("${path.module}/../../docker/Dockerfile")
    dockerignore_txt    = file("${path.module}/../../docker/.dockerignore")
    requirements_txt    = file("${path.module}/../../requirements.txt")
    fraud_script_py     = file("${path.module}/../../scripts/fraud_detection_pyspark.py")
    dataset_bucket_name = google_storage_bucket.dataset.name
    dataset_object_name = google_storage_bucket_object.creditcard_dataset.name
  })

  depends_on = [
    google_storage_bucket_object.creditcard_dataset,
    google_storage_bucket_iam_member.dataset_reader,
  ]
}
```

Several Session 02_02 concepts in one block:

- **Data source for the image.** `data "google_compute_image" "ubuntu"` resolves to "whatever Google currently considers the latest Ubuntu 22.04 LTS image". If you hardcoded an image ID, your config would rot the moment Google patched it. Slide 02_02 p14 is explicit about this.
- **`access_config {}`** with no body is the GCP idiom for "give me an ephemeral public IP". Drop the block and the VM has no internet-facing IP at all (useful for production worker nodes).
- **`service_account` with a narrow `scopes`.** Combined with the IAM binding from Step 4, this is the full identity story. The VM authenticates as `fraud-vm-sa`, and `fraud-vm-sa` can only read the dataset bucket.
- **`metadata_startup_script` + `templatefile()`.** This is the trick that ties the file system together. Terraform reads `startup.sh` *as a template*, substitutes `${dockerfile_txt}` etc. with the contents of the actual Dockerfile and Python script, and ships the entire thing to the VM as the cloud-init startup script. The VM never needs to clone the repo — it boots with the files baked in.
- **`depends_on`.** The implicit dependencies above are resource-attribute references. But the IAM binding does not feed any value into the VM, so we make the dependency explicit. Without this, Terraform might create the VM before the IAM grant exists, the startup script would `curl` the dataset before the VM had permission, and the deploy would race.

Now open [infra/terraform/startup.sh](infra/terraform/startup.sh). The key block is the dataset download:

```bash
ENCODED_OBJECT="$(jq -rn --arg value "$DATA_OBJECT" '$value|@uri')"
ACCESS_TOKEN="$(curl --fail --silent --show-error -H "Metadata-Flavor: Google" "http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/token" | jq -r '.access_token')"
curl --fail --location --silent --show-error \
  -H "Authorization: Bearer $ACCESS_TOKEN" \
  -o "$DATA_PATH" \
  "https://storage.googleapis.com/storage/v1/b/$DATA_BUCKET/o/$ENCODED_OBJECT?alt=media"
run_fraud_detection.sh
```

This is the *cloud-native authentication idiom* and worth understanding in detail:

1. **`http://metadata.google.internal`** is a magic IP (169.254.169.254) that GCE intercepts at the hypervisor level. Only code running *inside* a GCE VM can reach it. The required `Metadata-Flavor: Google` header proves the request is intentional, not a misrouted external request.
2. The metadata server returns an OAuth 2.0 access token tied to the VM's attached service account. We did not ship any credential file, password, or key. The VM proves its identity by *being itself*.
3. We use that token as a `Bearer` header to call the GCS JSON API and stream the object to disk.

This pattern works identically on AWS (`http://169.254.169.254/latest/meta-data/iam/security-credentials/...`) and Azure (with `Metadata: true`), so the concept transfers across clouds.


## Step 6 — Wire it together with variables and outputs

Hardcoding values in `main.tf` is brittle. [infra/terraform/variables.tf](infra/terraform/variables.tf) parameterises everything users might want to override:

```hcl
variable "machine_type" {
  description = "Machine type for the VM."
  type        = string
  default     = "e2-medium"
}

variable "ssh_source_ranges" {
  description = "CIDR ranges allowed to connect over SSH."
  type        = list(string)
  default     = ["0.0.0.0/0"]
}
```

[infra/terraform/outputs.tf](infra/terraform/outputs.tf) exposes the values you will need *after* applying:

```hcl
output "external_ip" {
  description = "Ephemeral public IP assigned to the VM."
  value       = google_compute_instance.ml_vm.network_interface[0].access_config[0].nat_ip
}

output "gcloud_ssh_command" {
  description = "Convenient SSH command for the provisioned VM."
  value       = "gcloud compute ssh ${google_compute_instance.ml_vm.name} --zone ${google_compute_instance.ml_vm.zone}"
}
```

This is the pattern from Session 03_02 slide 13: outputs are how Terraform communicates back to the human (or the next tool in your CI pipeline) what it just built.

Copy the example tfvars into a real one and edit it:

```bash
cp infra/terraform/terraform.tfvars.example infra/terraform/terraform.tfvars
```

Then open `infra/terraform/terraform.tfvars` and set at least:

- `project_id` — your GCP project ID.
- `ssh_source_ranges` — replace the placeholder with `["YOUR.PUBLIC.IP/32"]`. Find your public IP at https://api.ipify.org.

`terraform.tfvars` is gitignored. The `.example` file is committed as a template.


## Step 7 — Run it

```bash
cd infra/terraform
terraform init       # downloads the google provider plugin (Session 01_02 slide 15)
terraform fmt        # canonical formatting
terraform validate   # syntax + reference check
terraform plan       # dry-run: shows every resource that will be created
terraform apply      # type 'yes' when prompted
```

The `apply` step takes 2–4 minutes. In order, Terraform will:

1. Create the VPC, subnet, and firewall rule.
2. Create the GCS bucket.
3. Upload `data/creditcard.csv` (this takes the longest — 150 MB to upload).
4. Create the service account.
5. Bind the service account to the bucket with `objectViewer`.
6. Boot the VM with the startup script attached.

Terraform finishes once the VM is created. The startup script then runs *on the VM*, asynchronously: it installs Docker, downloads the dataset from GCS, builds the image, and runs the container. This typically takes another 5–8 minutes.


## Step 8 — Verify the workload ran

Connect to the VM:

```bash
$(terraform output -raw gcloud_ssh_command)
```

Once on the VM:

```bash
# What did the startup script do?
cat /opt/vm-ml-tutorial/startup.log

# What got staged?
ls -la /opt/vm-ml-tutorial

# What did the container produce?
ls -la /opt/vm-ml-tutorial/output
cat /opt/vm-ml-tutorial/output/summary.json
head /opt/vm-ml-tutorial/output/sample_predictions.csv
tail -50 /opt/vm-ml-tutorial/output/run.log
```

`summary.json` should show something like:

```json
{
  "rows_before_cleaning": 284807,
  "rows_after_cleaning": 283726,
  "train_rows": 226997,
  "test_rows": 56729,
  "area_under_roc": 0.9742,
  "sql_summary": [...],
  "confusion_matrix": [...]
}
```

An AUC of around 0.97 is the expected ballpark for a vanilla logistic regression on this dataset. The confusion matrix will be heavily skewed toward the negative class because fraud is only about 0.17% of the data.


## Step 9 — Re-run and clean up

The container is rerunnable. Any time you want fresh output (e.g. after editing the script and re-templating with `terraform apply`):

```bash
# On the VM
run_fraud_detection.sh
```

This rebuilds the image (Docker's layer cache makes the rebuild fast), removes the old container, runs a fresh one, and tees stdout to `/opt/vm-ml-tutorial/output/run.log`.

When you are done, **destroy everything**:

```bash
cd infra/terraform
terraform destroy
```

Type `yes`. Terraform reads the state file it created during `apply`, walks the dependency graph in reverse, and deletes the VM, the IAM binding, the service account, the dataset object, the bucket, the firewall rule, the subnet, and the VPC. You should see a "Destroy complete! Resources: N destroyed" line.

> **Always destroy.** A forgotten `e2-medium` VM costs roughly £20/month. A forgotten GCS bucket with 150 MB costs pennies but counts toward your project quota.


## Recap: what you learned

| Concept (slide reference) | Where it appears in this tutorial |
|---|---|
| IaC: declarative vs imperative (01_02 p6) | Step 5 — implicit dependencies |
| `init / plan / apply / destroy` (01_02 p15) | Step 7, Step 9 |
| Custom-mode VPC (02_02 p7) | Step 2 |
| Firewall with `target_tags` (02_02 p8) | Step 2 |
| Data source for image lookup (02_02 p14) | Step 5 |
| Implicit vs explicit dependencies (02_02 p11) | Step 5 — `depends_on` for IAM |
| `variables.tf` / `outputs.tf` / `tfvars` (03_02 p15-16) | Step 6 |
| Dockerfile layer caching (02_01 p8) | Step 1 |
| `.dockerignore` (02_01 p22) | Step 1 |
| `CMD` exec form (02_01 p17) | Step 1 |
| `SparkSession` / `read.csv` (01_03 p12-13) | Step 0 / payload |
| `na.drop().dropDuplicates() / cache()` (02_03 p5,12) | Step 0 / payload |
| `VectorAssembler / Pipeline` (04_03 p10,12) | Step 0 / payload |
| `BinaryClassificationEvaluator` (04_03 p15) | Step 8 — the AUC in `summary.json` |


## Concepts beyond the slides

This tutorial also touches a few patterns the lectures introduce only at a conceptual level:

- **GCE metadata-server token exchange** — Step 5 — instead of shipping a service-account JSON key onto the VM.
- **Scoped bucket-level IAM binding** — Step 4 — instead of project-wide grants.
- **`templatefile()` to compose a startup script from many source files** — Step 5.
- **`uniform_bucket_level_access`** — Step 3 — modern GCS best practice.
- **Read-only volume mount** — `-v "$WORKDIR/data:/app/data:ro"` in `startup.sh`.


## Where to go next

Natural follow-up tutorials inside this module's syllabus:

- **Terraform State (Session 04_02)** and **Modules (Session 05_02)** — moving the dataset bucket out into a reusable module and storing state in a remote backend so a team can collaborate.
- **CI/CD with Terraform (Session 06_01)** — running `terraform plan` automatically on every PR.
- **Multi-stage Docker builds and non-root users (02_01 p25-27)** — shrinking the image and tightening container security.
- **Spark Streaming (Sessions 05_03 / 06_03)** — turning this batch fraud-detection job into a near-real-time pipeline.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `terraform apply` errors with `Invalid value for "name"` on the VM | `instance_name` contains an underscore or uppercase letter | GCP VM names must match `[a-z]([-a-z0-9]*[a-z0-9])?` — use hyphens |
| `terraform validate` errors with "no file exists at .../creditcard.csv" | Dataset is missing locally | Download `creditcard.csv` from Kaggle into `data/` |
| SSH times out | Your public IP changed, or `ssh_source_ranges` is wrong | Re-check `https://api.ipify.org` and update `terraform.tfvars`, then `terraform apply` |
| `summary.json` never appears on the VM | Startup script is still running or failed | `cat /var/log/syslog \| grep startup-script` on the VM |
| Container build fails locally with `UnsupportedClassVersionError` | Your laptop has Java 8; PySpark 4 needs Java 17 | Install JDK 17 from https://adoptium.net, or just run inside Docker |


In [ ]:
# Load and display the summary.json produced by the cloud workload.
# Pull it back from the VM with:
#   gcloud compute scp <vm>:/opt/fraud-detection/output/summary.json ./output/summary.json --zone <zone>
import json
from pathlib import Path

summary_path = Path("output/summary.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2))
else:
    print(f"{summary_path} not found yet -- run the cloud workload and copy the file back first.")